In [1]:

import numpy as np
import pandas as pd
import torch
import folium
from IPython.display import display

from lstm import lstm_seq2seq


In [2]:

# =========================
# CONFIG
# =========================
TRAIN_CSV = "../data/vessel_prediction_model_data/mid_model_data/mid_train_data.csv"
VAL_CSV   = "../data/vessel_prediction_model_data/mid_model_data/mid_val_data.csv"
TEST_CSV  = "../data/vessel_prediction_model_data/mid_model_data/mid_test_data.csv"

SEQ_LEN = 10
TARGET_LEN = 1

BATCH_SIZE = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2
LEARNING_RATE = 0.001
MAX_EPOCHS = 100
TEACHER_FORCING_RATIO = 0.7

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH = "best_lstm_seq2seq_dlat_dlon.pt"

# Set to True only if you want to retrain inside this notebook.
TRAIN_MODEL = False

feature_cols = [
    "LAT",
    "LON",
    "SPEED",
    "dt",
    "cog_sin",
    "cog_cos",
    "hdg_sin",
    "hdg_cos",
    "dlat",
    "dlon",
]

target_cols = ["dlat", "dlon"]
route_col = "voyage_id"
time_col = "TIME"

YARDS_PER_METER = 1.0936132983377078


In [3]:

# =========================
# LOAD RAW DATA
# =========================
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

display(test_df.head())


train: (34125, 18)
val:   (9159, 18)
test:  (3356, 18)


,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME,dt,dlat,dlon,cog_rad,cog_sin,cog_cos,hdg_rad,hdg_sin,hdg_cos
0,205111000,22.697428,-78.498302,2023-09-01 14:10:00,13.2,108.4,107.0,1_205111000,2023-09-01 14:10:00,0.0,0.000000,0.000000,1.891937,0.948876,-0.315649,1.867502,0.956305,-0.292372
1,205111000,22.694939,-78.490204,2023-09-01 14:15:00,13.2,108.4,107.0,1_205111000,2023-09-01 14:15:00,300.0,-0.002489,0.008098,1.891937,0.948876,-0.315649,1.867502,0.956305,-0.292372
2,205111000,22.681163,-78.446551,2023-09-01 14:25:00,13.2,109.3,108.0,1_205111000,2023-09-01 14:25:00,600.0,-0.013776,0.043653,1.907645,0.943801,-0.330514,1.884956,0.951057,-0.309017
3,205111000,22.645493,-78.339608,2023-09-01 14:55:00,13.6,109.8,110.0,1_205111000,2023-09-01 14:55:00,1800.0,-0.035670,0.106943,1.916372,0.940881,-0.338738,1.919862,0.939693,-0.342020
4,205111000,22.637551,-78.316039,2023-09-01 15:00:00,13.7,108.9,110.0,1_205111000,2023-09-01 15:00:00,300.0,-0.007942,0.023569,1.900664,0.946085,-0.323917,1.919862,0.939693,-0.342020


In [4]:

# =========================
# WINDOWING BY VOYAGE_ID
# =========================
def make_windows_from_df(
    df: pd.DataFrame,
    feature_cols,
    target_cols,
    seq_len,
    target_len,
    route_col="voyage_id",
    time_col="TIME",
):
    """
    Returns
    -------
    X : torch.FloatTensor
        Shape (seq_len, N, n_features)
    Y : torch.FloatTensor
        Shape (target_len, N, n_targets)
    meta_df : pd.DataFrame
        One row per window with route/sample metadata
    """
    X_list = []
    Y_list = []
    meta_rows = []

    work = df.copy()
    work[time_col] = pd.to_datetime(work[time_col])

    for route_id, g in work.groupby(route_col, sort=False):
        g = g.sort_values(time_col).reset_index(drop=True)

        feats = g[feature_cols].to_numpy(dtype=np.float32)
        targs = g[target_cols].to_numpy(dtype=np.float32)
        raw_latlon = g[["LAT", "LON"]].to_numpy(dtype=np.float64)
        raw_times = g[time_col].to_numpy()

        n = len(g)
        if n < seq_len + target_len:
            continue

        for start in range(n - (seq_len + target_len) + 1):
            hist_end = start + seq_len
            fut_end = hist_end + target_len

            X_list.append(feats[start:hist_end])
            Y_list.append(targs[hist_end:fut_end])

            meta_rows.append(
                {
                    "voyage_id": route_id,
                    "window_start": start,
                    "history_start_time": raw_times[start],
                    "history_end_time": raw_times[hist_end - 1],
                    "target_time": raw_times[hist_end],
                    "history_latlon": raw_latlon[start:hist_end].copy(),
                    "true_future_latlon": raw_latlon[hist_end:fut_end].copy(),
                }
            )

    if not X_list:
        raise ValueError("No windows created. Reduce seq_len/target_len or inspect the data.")

    X = torch.from_numpy(np.stack(X_list, axis=0)).permute(1, 0, 2).contiguous()
    Y = torch.from_numpy(np.stack(Y_list, axis=0)).permute(1, 0, 2).contiguous()
    meta_df = pd.DataFrame(meta_rows)

    return X, Y, meta_df


X_train, Y_train, meta_train = make_windows_from_df(
    train_df, feature_cols, target_cols, seq_len=SEQ_LEN, target_len=TARGET_LEN,
    route_col=route_col, time_col=time_col
)
X_val, Y_val, meta_val = make_windows_from_df(
    val_df, feature_cols, target_cols, seq_len=SEQ_LEN, target_len=TARGET_LEN,
    route_col=route_col, time_col=time_col
)
X_test, Y_test, meta_test = make_windows_from_df(
    test_df, feature_cols, target_cols, seq_len=SEQ_LEN, target_len=TARGET_LEN,
    route_col=route_col, time_col=time_col
)

print("X_train:", tuple(X_train.shape), "Y_train:", tuple(Y_train.shape))
print("X_val:  ", tuple(X_val.shape), "Y_val:  ", tuple(Y_val.shape))
print("X_test: ", tuple(X_test.shape), "Y_test: ", tuple(Y_test.shape))

display(meta_test[["voyage_id", "window_start", "history_end_time", "target_time"]].head())


X_train: (10, 15391, 10) Y_train: (1, 15391, 2)
X_val:   (10, 4050, 10) Y_val:   (1, 4050, 2)
X_test:  (10, 1538, 10) Y_test:  (1, 1538, 2)


,voyage_id,window_start,history_end_time,target_time
0,5_209425000,0,2024-02-13 13:50:00,2024-02-13 13:55:00
1,5_209425000,1,2024-02-13 13:55:00,2024-02-13 14:00:00
2,5_209425000,2,2024-02-13 14:00:00,2024-02-13 14:05:00
3,5_209425000,3,2024-02-13 14:05:00,2024-02-13 14:15:00
4,5_209425000,4,2024-02-13 14:15:00,2024-02-13 14:20:00


In [5]:
# =========================
# MODEL
# =========================
model = lstm_seq2seq(
    input_size=len(feature_cols),
    hidden_size=HIDDEN_SIZE,
    output_size=2,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    decoder_feature_indices=tuple(range(len(feature_cols))),
).to(DEVICE)

history = model.train_model(
        input_tensor=X_train,
        target_tensor=Y_train,
        target_len=TARGET_LEN,
        batch_size=BATCH_SIZE,
        val_input_tensor=X_val,
        val_target_tensor=Y_val,
        max_epochs=MAX_EPOCHS,
        training_prediction="mixed_teacher_forcing",
        teacher_forcing_ratio=TEACHER_FORCING_RATIO,
        learning_rate=LEARNING_RATE,
        dynamic_tf=True,
        patience=10,
        min_delta=1e-5,
        save_path=SAVE_PATH,
        grad_clip=1.0,
        shuffle_batches=True,
)
print("Training done.")
print("Best epoch:", history["best_epoch"])
print("Best val loss:", history["best_val_loss"])


model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()
print("Model ready.")


 22%|██▏       | 22/100 [00:27<01:38,  1.26s/it, no_improve=10, train=0.0003, val=0.0003]

Training done.
Best epoch: 12
Best val loss: 0.00026054615501891476
Model ready.



C:\Users\logan\AppData\Local\Temp\ipykernel_14872\693167991.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(SAVE_PATH, map_location=DE

In [6]:

# =========================
# HELPERS
# =========================
def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def predict_dataset_deltas(model, X_tensor, target_len=1, batch_size=256):
    model.eval()
    n_samples = X_tensor.shape[1]
    preds = []

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        x_batch = X_tensor[:, start:end, :].to(DEVICE)

        batch_pred = model.predict(
            x_batch,
            target_len=target_len,
        )

        preds.append(batch_pred)

    return np.concatenate(preds, axis=1)

def deltas_to_absolute(history_last_abs, delta_arr):
    """
    history_last_abs: (N, 2)
    delta_arr: (target_len, N, 2)
    returns: (target_len, N, 2)
    """
    current = history_last_abs.astype(np.float64).copy()
    abs_list = []

    for t in range(delta_arr.shape[0]):
        current = current + delta_arr[t]
        abs_list.append(current.copy())

    return np.stack(abs_list, axis=0)

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return R * c

def metrics_yards(pred_abs, true_abs):
    """
    pred_abs, true_abs: numpy arrays of shape (target_len, N, 2)
    Computes point-to-point geodesic error in yards.
    """
    err_m = haversine_m(
        true_abs[..., 0],
        true_abs[..., 1],
        pred_abs[..., 0],
        pred_abs[..., 1],
    )
    err_yd = err_m * YARDS_PER_METER

    mae_yd = np.mean(np.abs(err_yd))
    mse_yd2 = np.mean(err_yd ** 2)
    rmse_yd = np.sqrt(mse_yd2)

    return mae_yd, mse_yd2, rmse_yd, err_yd

def dead_reckoning_from_history(X_np, target_len, feature_cols):
    """
    Uses last observed speed/course and dt to extrapolate future absolute lat/lon.
    Returns array of shape (target_len, N, 2).
    """
    idx_lat = feature_cols.index("LAT")
    idx_lon = feature_cols.index("LON")
    idx_speed = feature_cols.index("SPEED")
    idx_dt = feature_cols.index("dt")
    idx_cog_sin = feature_cols.index("cog_sin")
    idx_cog_cos = feature_cols.index("cog_cos")

    last = X_np[-1]  # (N, F)

    lat = last[:, idx_lat].astype(np.float64)
    lon = last[:, idx_lon].astype(np.float64)
    speed_kn = last[:, idx_speed].astype(np.float64)
    dt_seconds = last[:, idx_dt].astype(np.float64)
    cog = np.arctan2(last[:, idx_cog_sin], last[:, idx_cog_cos]).astype(np.float64)

    out = []
    cur_lat = lat.copy()
    cur_lon = lon.copy()

    for _ in range(target_len):
        distance_m = speed_kn * 0.514444 * dt_seconds
        d_north = distance_m * np.cos(cog)
        d_east = distance_m * np.sin(cog)

        dlat = d_north / 111320.0
        dlon = d_east / (111320.0 * np.cos(np.radians(cur_lat)))

        cur_lat = cur_lat + dlat
        cur_lon = cur_lon + dlon

        out.append(np.column_stack([cur_lat.copy(), cur_lon.copy()]))

    return np.stack(out, axis=0)

def mapmake(history, prediction, truth=None, center=(22.5, -77.8), zoom_start=8):
    m = folium.Map(location=list(center), zoom_start=zoom_start)

    history_coords = [(float(row[0]), float(row[1])) for row in history]
    pred_coords = [(float(row[0]), float(row[1])) for row in prediction]

    folium.PolyLine(history_coords, weight=3, tooltip="History").add_to(m)

    folium.Marker(
        location=list(history_coords[-1]),
        popup="Last ping",
        tooltip="Last ping"
    ).add_to(m)

    folium.PolyLine(
        pred_coords,
        weight=3,
        dash_array="5, 8",
        tooltip="Predicted"
    ).add_to(m)

    if truth is not None:
        truth_coords = [(float(row[0]), float(row[1])) for row in truth]
        folium.PolyLine(
            truth_coords,
            weight=3,
            dash_array="2, 6",
            tooltip="Actual"
        ).add_to(m)

        for i, (lat, lon) in enumerate(truth_coords, start=1):
            folium.CircleMarker(
                location=[lat, lon],
                radius=4,
                popup=f"Actual {i}",
                tooltip=f"Actual {i}",
                fill=True
            ).add_to(m)

    for i, (lat, lon) in enumerate(history_coords):
        folium.CircleMarker(
            location=[lat, lon],
            radius=3,
            popup=f"History {i}",
            fill=True
        ).add_to(m)

    for i, (lat, lon) in enumerate(pred_coords, start=1):
        folium.Marker(
            location=[lat, lon],
            popup=f"Predicted {i}",
            tooltip=f"Predicted {i}"
        ).add_to(m)

    return m


In [7]:

# =========================
# PREP TEST ARRAYS
# =========================
X_test_np = X_test.detach().cpu().numpy().copy()
Y_test_np = Y_test.detach().cpu().numpy().copy()

# Last observed absolute lat/lon from history window
lat_idx = feature_cols.index("LAT")
lon_idx = feature_cols.index("LON")

history_abs_all = X_test_np[:, :, [lat_idx, lon_idx]].astype(np.float64).copy()   # (seq_len, N, 2)
history_last_abs = history_abs_all[-1, :, :]                                       # (N, 2)

print("history_abs_all shape:", history_abs_all.shape)
print("history_last_abs shape:", history_last_abs.shape)
print("example last history point:", history_last_abs[0])


history_abs_all shape: (10, 1538, 2)
history_last_abs shape: (1538, 2)
example last history point: [ 21.78906059 -76.96216583]


In [8]:

# =========================
# LSTM TEST METRICS IN YARDS
# =========================
Y_pred_delta = predict_dataset_deltas(model, X_test, target_len=TARGET_LEN, batch_size=256)
Y_true_delta = Y_test_np.copy()

print("Y_pred_delta shape:", Y_pred_delta.shape)
print("Y_true_delta shape:", Y_true_delta.shape)
print("example predicted delta:", Y_pred_delta[0, 0])
print("example true delta:     ", Y_true_delta[0, 0])

pred_future_abs = deltas_to_absolute(history_last_abs, Y_pred_delta)
true_future_abs = deltas_to_absolute(history_last_abs, Y_true_delta)

print("example predicted abs point:", pred_future_abs[0, 0])
print("example true abs point:     ", true_future_abs[0, 0])

mae_yd, mse_yd2, rmse_yd, err_yd = metrics_yards(pred_future_abs, true_future_abs)

print("LSTM test metrics in yards")
print(f"MAE  (yd): {mae_yd:,.3f}")
print(f"MSE  (yd²): {mse_yd2:,.3f}")
print(f"RMSE (yd): {rmse_yd:,.3f}")


Y_pred_delta shape: (1, 1538, 2)
Y_true_delta shape: (1, 1538, 2)
example predicted delta: [ 0.01351526 -0.02859255]
example true delta:      [ 0.009857 -0.024231]
example predicted abs point: [ 21.80257585 -76.99075839]
example true abs point:      [ 21.79891759 -76.98639683]
LSTM test metrics in yards
MAE  (yd): 1,406.387
MSE  (yd²): 5,597,269.197
RMSE (yd): 2,365.855


In [9]:
# =========================
# DEAD-RECKONING BASELINE
# =========================
dr_future_abs = dead_reckoning_from_history(X_test_np, target_len=TARGET_LEN, feature_cols=feature_cols)

dr_mae_yd, dr_mse_yd2, dr_rmse_yd, dr_err_yd = metrics_yards(dr_future_abs, true_future_abs)

print("Dead-reckoning test metrics in yards")
print(f"MAE  (yd): {dr_mae_yd:,.3f}")
print(f"MSE  (yd²): {dr_mse_yd2:,.3f}")
print(f"RMSE (yd): {dr_rmse_yd:,.3f}")


Dead-reckoning test metrics in yards
MAE  (yd): 1,380.416
MSE  (yd²): 8,195,266.707
RMSE (yd): 2,862.738


In [10]:

# =========================
# SINGLE-SAMPLE DEBUG
# =========================
sample_idx = 0

print("voyage_id:", meta_test.iloc[sample_idx]["voyage_id"])
print("history end time:", meta_test.iloc[sample_idx]["history_end_time"])
print("target time:", meta_test.iloc[sample_idx]["target_time"])
print("last history point:", history_last_abs[sample_idx])
print("pred delta:", Y_pred_delta[0, sample_idx])
print("true delta:", Y_true_delta[0, sample_idx])
print("pred abs:", pred_future_abs[0, sample_idx])
print("true abs:", true_future_abs[0, sample_idx])

sample_err_m = haversine_m(
    true_future_abs[0, sample_idx, 0],
    true_future_abs[0, sample_idx, 1],
    pred_future_abs[0, sample_idx, 0],
    pred_future_abs[0, sample_idx, 1],
)
print("sample error (yards):", sample_err_m * YARDS_PER_METER)


voyage_id: 5_209425000
history end time: 2024-02-13 13:50:00
target time: 2024-02-13 13:55:00
last history point: [ 21.78906059 -76.96216583]
pred delta: [ 0.01351526 -0.02859255]
true delta: [ 0.009857 -0.024231]
pred abs: [ 21.80257585 -76.99075839]
true abs: [ 21.79891759 -76.98639683]
sample error (yards): 663.6325039782467


In [11]:

# =========================
# MAP A FEW TEST VOYAGES
# =========================
unique_test_voyages = meta_test["voyage_id"].drop_duplicates().tolist()[:3]

for map_num, voyage_id in enumerate(unique_test_voyages, start=1):
    row = meta_test[meta_test["voyage_id"] == voyage_id].iloc[0]
    sample_idx = row.name

    history_abs = history_abs_all[:, sample_idx, :]      # (seq_len, 2)
    pred_point_abs = pred_future_abs[:, sample_idx, :]   # (target_len, 2)
    true_point_abs = true_future_abs[:, sample_idx, :]   # (target_len, 2)

    print(f"Map {map_num} | voyage_id={voyage_id}")
    print("History last AIS point:  ", history_abs[-1])
    print("Predicted next AIS point:", pred_point_abs[0])
    print("True next AIS point:     ", true_point_abs[0])

    display(mapmake(history_abs, pred_point_abs, truth=true_point_abs))
    print("-" * 70)


Map 1 | voyage_id=5_209425000
History last AIS point:   [ 21.78906059 -76.96216583]
Predicted next AIS point: [ 21.80257585 -76.99075839]
True next AIS point:      [ 21.79891759 -76.98639683]


----------------------------------------------------------------------
Map 2 | voyage_id=8_209425000
History last AIS point:   [ 21.79094315 -76.9582901 ]
Predicted next AIS point: [ 21.80383582 -76.98595906]
True next AIS point:      [ 21.80125515 -76.9832881 ]


----------------------------------------------------------------------
Map 3 | voyage_id=9_209425000
History last AIS point:   [ 22.20437431 -77.49827576]
Predicted next AIS point: [ 22.22158238 -77.51886239]
True next AIS point:      [ 22.21840231 -77.51426376]


----------------------------------------------------------------------
